# Mini-Project: Medallion ETL (raw to bronze to silver)

**Module:** Week 3 Day 3 pandas review and medallion ETL practice  
**Estimated time:** 2 to 3 hours including the optional cloud attempt  
**Format:** Individual, pair help encouraged  
**Kernel:** the shared repo-root `.venv`  
**AI-Free Zone:** write the code yourself.

You are building a small, real pipeline, one layer at a time:

```text
raw weather CSV
  -> bronze: the raw bytes landed unchanged
  -> silver: cleaned, typed, deduplicated, written as parquet
```

Bronze answers "what did we receive?" Silver answers "what can analysts trust?" Those two must never be the same file.


## Definition of Done

By the end of the required part, you should have:

1. A bronze CSV copied byte-for-byte into a dated folder.
2. A pandas silver DataFrame with 14 rows.
3. A silver parquet file written and read back successfully.
4. A short evidence note explaining what worked.

Optional after that:

- Try the same silver logic in Polars.
- Attempt S3 to GCS.
- Convert the notebook into a script after instructor approval.

Local pipeline success is required. Cloud success is optional. A serious cloud attempt plus a clear blocker note is acceptable evidence.


In [ ]:
import io
from datetime import date
from pathlib import Path

import pandas as pd

RAW_PATH = Path("data/raw/weather_raw.csv")
BRONZE_DIR = Path("data/bronze")
SILVER_DIR = Path("data/silver")
DATASET = "weather"
INGEST_DATE = date.today().isoformat()

print("raw file exists:", RAW_PATH.exists())
print("ingest date:", INGEST_DATE)


## Before You Start: Three Ideas Used Immediately

This project uses a few ideas that feel more like data engineering than pure pandas:

| Idea | Plain meaning | Where it appears |
|---|---|---|
| `bytes` | The exact raw file contents before pandas interprets them | Bronze |
| `io.BytesIO` | A wrapper that lets pandas read bytes as if they were a file | Profiling and silver |
| `assert` | A checkpoint that stops the notebook if something important is false | Bronze proof and parquet proof |

Run the example cells below before the assignment cells. They are small on purpose.


### Example 1: Text, Bytes, `StringIO`, and `BytesIO`

`StringIO` is for text that is already a Python string. `BytesIO` is for raw bytes, like the result of `Path.read_bytes()` or an S3 object body.

Bronze stores bytes because the bronze layer is evidence. Silver uses pandas because the silver layer is an interpreted table.


In [ ]:
sample_csv_text = "station,date,tmax\nBOS,2026-07-01,82\nPVD,2026-07-01,79\n"

# Text path: pandas reads a string through StringIO.
text_df = pd.read_csv(io.StringIO(sample_csv_text))
print("StringIO rows:", len(text_df))

# Bytes path: encode text into bytes, then pandas reads bytes through BytesIO.
sample_csv_bytes = sample_csv_text.encode("utf-8")
bytes_df = pd.read_csv(io.BytesIO(sample_csv_bytes))
print("BytesIO rows:", len(bytes_df))
print("bytes are raw evidence:", sample_csv_bytes[:20])


### Example 2: `assert` as a Trust Check

Use `assert` when a condition must be true for the pipeline to be trusted. If the condition is false, Python raises an `AssertionError` and stops that cell.

In this project, `assert` helps prove two things:

1. Bronze matches the raw source bytes.
2. The parquet file you wrote can be read back with the expected shape.


In [ ]:
expected_rows = 2
actual_rows = len(bytes_df)
assert actual_rows == expected_rows, f"expected {expected_rows} rows, got {actual_rows}"
print("assert passed: the example row count is trusted")


# Tier 1: Required Core Journey

The required journey is local and pandas-first:

1. Read raw CSV as bytes.
2. Write bronze unchanged.
3. Prove bronze is byte-for-byte identical.
4. Load raw bytes into pandas.
5. Profile the data.
6. Apply silver cleaning policies one at a time.
7. Write silver parquet.
8. Read silver parquet back and assert the shape.
9. Write a short evidence summary.


## Part 1: Extract, as Bytes

Read the local raw CSV as bytes, not as a DataFrame.

Why: bronze must preserve exactly what arrived. The moment pandas parses a file, it starts interpreting types, encodings, and missing values. That interpretation belongs in silver, not bronze.


In [ ]:
# TODO: read RAW_PATH as raw bytes with Path.read_bytes() and store it in raw_bytes.
# TODO: print how many bytes you received.

# Example shape:
# raw_bytes = RAW_PATH.read_bytes()
# print("extracted", len(raw_bytes), "bytes")


## Part 2: Bronze, Landed Unchanged

Write the bytes to the bronze layer using a dated folder, the same convention real lakes use:

```text
data/bronze/weather/ingest_date=YYYY-MM-DD/weather_raw.csv
```

The `ingest_date=` folder name is not decoration. Tomorrow you will see BigQuery use the same `key=value` idea for partition-style paths.


In [ ]:
# TODO: build the bronze folder path.
# TODO: create the folder with mkdir(parents=True, exist_ok=True).
# TODO: print bronze_dir.

# Example shape:
# bronze_dir = BRONZE_DIR / DATASET / f"ingest_date={INGEST_DATE}"
# bronze_dir.mkdir(parents=True, exist_ok=True)
# print("bronze folder:", bronze_dir)


In [ ]:
# TODO: build bronze_path for the raw CSV file.
# TODO: write raw_bytes to bronze_path with Path.write_bytes().
# TODO: print bronze_path.

# Example shape:
# bronze_path = bronze_dir / f"{DATASET}_raw.csv"
# bronze_path.write_bytes(raw_bytes)
# print("bronze file:", bronze_path)


### Bronze Proof with `assert`

This proof matters more than it looks. It answers: "How do we know bronze was not modified?"

The check reads the bronze file back as bytes and compares it to the bytes you extracted. If they match, bronze is byte-for-byte identical to the source.


In [ ]:
# TODO: run this after bronze_path exists.
assert bronze_path.read_bytes() == raw_bytes, "bronze changed from the raw source bytes"
print("bronze verified: byte-for-byte identical to the source")


## Part 3: Profile Before Cleaning

Look before you clean. Load the raw bytes into pandas, then inspect the data in small steps.

`pd.read_csv(io.BytesIO(raw_bytes))` means: "Treat these raw bytes like a readable file, then let pandas parse the CSV."


In [ ]:
raw_df = pd.read_csv(io.BytesIO(raw_bytes))
print("raw shape:", raw_df.shape)
raw_df.head()


In [ ]:
# TODO: inspect column data types.
# Question: which columns should become numeric or dates later?

raw_df.dtypes


In [ ]:
# TODO: inspect missing values by column.

raw_df.isna().sum()


In [ ]:
# TODO: count duplicated (STATION, DATE) pairs.

# Example shape:
# raw_df.duplicated(subset=["STATION", "DATE"]).sum()


## Part 4: Silver with pandas, One Policy per Cell

Silver is where interpretation happens. Each policy below should be explicit enough that you can defend it.

You do not need to solve it exactly like the solution notebook. What matters is that your code applies the required policies clearly and produces the expected evidence.


### Intermediate Checkpoints

Use this table while debugging. If a later number is wrong, go back to the last checkpoint that matched.

| Step | Expected evidence |
|---|---|
| After reading raw CSV | Shape is `(20, 5)` |
| After date parsing | `1` failed date |
| After dropping missing dates | `19` rows |
| After dropping duplicates | `17` rows |
| After dropping missing temperatures | `14` rows |
| Final silver | `14` rows, with BOS 5, PVD 3, NY 6 |


### Example: One Cleaning Policy with a Row Count

The pattern is:

1. Save `before = len(df)`.
2. Apply one policy.
3. Print how many rows remain and how many were dropped.

This makes your notebook explain itself.


In [ ]:
example_policy_df = pd.DataFrame({
    "station": [" bos ", "BOS", "PVD"],
    "date": ["2026-07-01", "not a date", "2026-07-01"],
})
example_policy_df["station"] = example_policy_df["station"].astype(str).str.strip().str.upper()
example_policy_df["date"] = pd.to_datetime(example_policy_df["date"], errors="coerce")

before = len(example_policy_df)
example_policy_df = example_policy_df.dropna(subset=["date"])
print("after date policy:", len(example_policy_df), "| dropped:", before - len(example_policy_df))
example_policy_df


### Policy 1: Normalize Column Names

Why this policy exists: lowercase column names make future code predictable. If one file says `TMAX` and another says ` tmax `, the silver table should expose one clean name.


In [ ]:
silver = raw_df.copy()

# TODO Policy 1: normalize column names to stripped lowercase.

silver.head()


### Policy 2: Clean Station IDs

Why this policy exists: station IDs are business keys. Values like `bos`, `BOS`, and ` BOS ` should become one value before you deduplicate or join.


In [ ]:
# TODO Policy 2: station -> stripped, uppercase text.

silver["station"].head()


### Policies 3 and 4: Parse Dates and Coerce Numbers

Why these policies exist: invalid dates cannot be trusted for time-based analysis, so convert failures to missing values first. Numeric columns should be real numbers, not strings like `bad` or `N/A`.


In [ ]:
# TODO Policy 3: parse date. Strip the text first, then pd.to_datetime(..., errors="coerce").
# TODO: print how many dates failed to parse. Expected: 1.

# Example shape:
# silver["date"] = pd.to_datetime(silver["date"].astype(str).str.strip(), errors="coerce")
# print("failed dates:", silver["date"].isna().sum())


In [ ]:
# TODO Policy 4: coerce tmax, tmin, prcp to numeric with errors="coerce".

# Example shape:
# for col in ("tmax", "tmin", "prcp"):
#     silver[col] = pd.to_numeric(silver[col], errors="coerce")

silver.dtypes


### Policy 5: Drop Rows with No Date

Why this policy exists: a weather reading without a valid date cannot support time-based analysis or partitioned loading.


In [ ]:
# TODO Policy 5: drop rows with no date.
# TODO: print the row count and rows dropped. Expected row count: 19.

before = len(silver)


### Policy 6: Drop Duplicate Station-Date Rows

Why this policy exists: one station should have one observation per day in the silver table. Duplicates make downstream counts and averages lie.


In [ ]:
# TODO Policy 6: drop duplicate (station, date) pairs, keep the first.
# TODO: print the row count and rows dropped. Expected row count: 17.

before = len(silver)


### Policy 7: Fix Precipitation Values

Why this policy exists: missing precipitation means no recorded rain for this lab, so use `0.0`. Negative precipitation is physically impossible, so clip it to zero.


In [ ]:
# TODO Policy 7: prcp -> fillna(0.0) and clip(lower=0).

# Example shape:
# silver["prcp"] = silver["prcp"].fillna(0.0).clip(lower=0)


### Policy 8: Drop Rows Missing Required Temperatures

Why this policy exists: if max or min temperature is missing, the row cannot support temperature analysis or the `temp_range` feature.


In [ ]:
# TODO Policy 8: drop rows missing tmax or tmin.
# TODO: print the row count and rows dropped. Expected row count: 14.

before = len(silver)


### Derived Columns

Why this policy exists: silver tables often add useful, trusted columns that analysts should not have to rebuild every time.


In [ ]:
# TODO: add two derived columns:
#   temp_range = tmax - tmin
#   is_rainy = prcp > 0
# TODO: reset the index and print the final row count.
# TODO: print station counts.

silver.head()


Expected checkpoint:

```text
silver rows: 14
station counts: {'MA1BOS': 5, 'RI1PVD': 3, 'US1NY': 6}
```

If your number differs, walk back up the cells and find which policy dropped a different count than expected. That walk is the skill.


## Part 5: Write Silver as Parquet, and Read It Back

Silver is a table for analysts, so it gets a columnar format, not CSV. Always read your own output back. A pipeline is more trustworthy when it checks what it wrote.

The `assert` here is a quick checkpoint: "The parquet file I wrote has the same shape as the DataFrame I intended to write."


In [ ]:
# TODO: build silver_path.
# TODO: create the parent folder.
# TODO: write silver to parquet with index=False.
# TODO: print silver_path.

# Example shape:
# silver_path = SILVER_DIR / DATASET / f"{DATASET}_clean.parquet"
# silver_path.parent.mkdir(parents=True, exist_ok=True)
# silver.to_parquet(silver_path, index=False)
# print("silver:", silver_path)


In [ ]:
# TODO: read back the parquet and assert the shape matches.
# Hint: silver_reloaded = pd.read_parquet(silver_path)

silver_reloaded = None  # TODO: replace with pd.read_parquet(silver_path)
assert silver_reloaded is not None, "TODO: read the parquet back into silver_reloaded"
assert silver_reloaded.shape == silver.shape, (
    f"shape mismatch: wrote {silver.shape}, read back {silver_reloaded.shape}"
)
print("parquet round-trip ok:", silver_reloaded.shape)


## Required Evidence Summary

Write a short note in your README or PR description:

- Where the bronze file landed.
- How you proved bronze was unchanged.
- How many rows silver has.
- Whether the parquet round trip passed.

This is the required finish line before any optional section.


# Tier 2: Optional Reinforcement


## Part 6 (Optional Stretch): The Same Silver in Polars

**Stop here unless your pandas version is complete.**

Polars is optional for this mini-project. The required bar is pandas silver = 14 rows and a verified parquet write. If you attempt Polars, translate the same cleaning policies. Do not invent new policies.


In [ ]:
# OPTIONAL STRETCH: leave silver_pl as None if you skip this section.
silver_pl = None

# If you choose the stretch, uncomment the import and complete the steps below.
# import polars as pl

# TODO optional:
#   read io.BytesIO(raw_bytes) with pl.read_csv
#   rename columns to stripped lowercase
#   station: cast Utf8, strip_chars, to_uppercase
#   date: str.strptime(pl.Date, "%Y-%m-%d", strict=False)
#   tmax/tmin/prcp: cast Float64 with strict=False
#   drop_nulls date, unique on (station, date) keep first
#   prcp: when null or < 0 then 0.0 otherwise prcp
#   drop_nulls tmax and tmin
#   add temp_range and is_rainy
# Store it in silver_pl and print its height.


In [ ]:
if silver_pl is not None:
    assert len(silver) == silver_pl.height, "pandas and Polars disagree: compare your policies step by step"
    print("pandas and Polars silver row counts match:", len(silver))
else:
    print("Polars stretch skipped. Required pandas silver rows:", len(silver))


# Tier 3: Real-World Extension


## Part 7: Cloud, Only After Local Works

Local pipeline success is required. Cloud success is optional. A serious cloud attempt plus a clear blocker note is acceptable evidence.

Cloud changes the endpoints, not the transform. You already proved the pipeline locally:

| Layer | Local version | Cloud version | Why it exists |
|---|---|---|---|
| Extract | `data/raw/weather_raw.csv` | Public S3 object | Get the raw arrival |
| Bronze | `data/bronze/.../weather_raw.csv` | GCS bronze object | Preserve raw bytes unchanged |
| Silver | `data/silver/.../weather_clean.parquet` | GCS silver object | Store the trusted table |

Run one cloud cell at a time. If S3 or GCS is blocked, keep the printed blocker note.


### Why Cloud Uses `try`, `except`, and `finally`

Cloud cells can fail for reasons outside your pandas code: missing packages, network issues, GCP authentication, bucket names, or IAM permissions.

Use this pattern:

- `try`: run the risky cloud step.
- `except`: print a useful blocker note if it fails.
- `finally`: print status that should appear either way.


In [ ]:
status = "not started"
try:
    pretend_response = {"Body": io.BytesIO(sample_csv_bytes)}
    example_cloud_bytes = pretend_response["Body"].read()
    status = "ok"
except Exception as exc:
    status = "blocked"
    print("read failed:", exc)
finally:
    print("finally ran, status:", status)


### 7.0 Before You Run the Cloud Cells

Work from your copied notebook in `student-work/week3/day3/`, not from the provided file under `Week 3/Labs/...`.

Your setup should already be complete from `Activity_0_UV_GCS_Setup.md`. From the repo root:

```bash
uv add "pandas>=3.0" polars pyarrow boto3 google-cloud-storage
```

Confirm the Google Cloud CLI exists:

```bash
gcloud --version
```

Then authenticate to your project:

```bash
gcloud auth application-default login
gcloud config set project YOUR_PROJECT_ID
```

The global `.venv` for the course lives at the repo root. Select the repo-root `.venv/bin/python` as the VS Code interpreter and notebook kernel before running the notebook.


### 7.1 Configure the Cloud Endpoints

There are two cloud locations in this activity:

- `SOURCE_BUCKET` and `SOURCE_KEY`: the public S3 object you read from.
- `GCP_PROJECT_ID` and `GCS_BUCKET`: your assigned Google Cloud project and your bucket.

Only edit the GCP values. The S3 source is shared for the class.


In [ ]:
# Public class source. Do not edit these unless the instructor changes the source.
SOURCE_BUCKET = "techcatalyst-de-2026"
SOURCE_KEY = "raw/weather/weather_raw.csv"

# Your Google Cloud destination. Replace both placeholders before uploading to GCS.
GCP_PROJECT_ID = "your-gcp-project-id"  # TODO: replace with your assigned project ID
GCS_BUCKET = "techcatalyst-de-2026-your-username-raw"  # TODO: replace your-username

print("S3 source:", f"s3://{SOURCE_BUCKET}/{SOURCE_KEY}")
print("GCS target:", f"gs://{GCS_BUCKET}")


### 7.2 Extract from Public S3

This cell does the same job as Part 1: produce raw bytes.

The new idea is `Config(signature_version=UNSIGNED)`. That tells `boto3` not to look for AWS credentials because this object is public and read-only.

The object body is a stream. Calling `.read()` turns that stream into bytes, the same kind of value you got from `Path.read_bytes()` locally.


In [ ]:
s3_raw_bytes = None
cloud_raw_bytes = raw_bytes

try:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config

    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
    response = s3.get_object(Bucket=SOURCE_BUCKET, Key=SOURCE_KEY)
    s3_raw_bytes = response["Body"].read()
    cloud_raw_bytes = s3_raw_bytes
    print("S3 extract ok:", len(s3_raw_bytes), "bytes")
    print("matches local fallback:", s3_raw_bytes == raw_bytes)
except Exception as exc:
    print("S3 extract did not work. The notebook will use local raw bytes instead.")
    print("Check boto3 install, internet access, bucket name, and object key.")
    print("Original error:", exc)
finally:
    print("cloud raw byte source:", "S3" if s3_raw_bytes is not None else "local fallback")


### 7.3 Connect to GCS with Application Default Credentials

This cell prepares the destination. It does not upload anything yet.

Important: `client.bucket(GCS_BUCKET)` creates a Python handle for that bucket name. The upload cells prove whether the bucket exists and whether your account can write to it.


In [ ]:
bucket = None
try:
    from google.cloud import storage

    if GCP_PROJECT_ID == "your-gcp-project-id":
        raise ValueError("Replace GCP_PROJECT_ID with your assigned Google Cloud project ID.")
    if "your-username" in GCS_BUCKET:
        raise ValueError("Replace GCS_BUCKET with your actual bucket name.")

    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    print("GCS bucket handle ready:", f"gs://{GCS_BUCKET}")
except Exception as exc:
    print("GCS setup is not ready. Cloud upload cells will be skipped until this is fixed.")
    print("Original error:", exc)
finally:
    print("GCS setup status:", "ready" if bucket is not None else "blocked")


### 7.4 Upload Bronze to GCS

Bronze uploads raw bytes, not a DataFrame.

The path includes `ingest_date=...` so each run lands in a dated folder. That makes reruns auditable and keeps raw data separate from cleaned data.


In [ ]:
bronze_blob_name = f"bronze/{DATASET}/ingest_date={INGEST_DATE}/{DATASET}_raw.csv"
bronze_uri = None

if bucket is None:
    print("Skipping bronze upload because the GCS bucket is not ready.")
else:
    try:
        bronze_blob = bucket.blob(bronze_blob_name)
        bronze_blob.upload_from_string(cloud_raw_bytes, content_type="text/csv")
        bronze_uri = f"gs://{GCS_BUCKET}/{bronze_blob_name}"
        print("Uploaded bronze raw bytes:", bronze_uri)
        print("bronze byte count:", len(cloud_raw_bytes))
    except Exception as exc:
        print("Bronze upload did not work.")
        print("Check bucket name, ADC login, and Storage Object Admin permission.")
        print("Original error:", exc)

print("bronze cloud status:", bronze_uri or "not uploaded")


### 7.5 Upload Silver to GCS

Silver uploads the parquet file you already wrote and verified in Part 5.

This is intentionally simpler than creating parquet bytes in memory. The local file is your evidence, and the cloud upload sends that same file to the silver object path.


In [ ]:
silver_blob_name = f"silver/{DATASET}/{DATASET}_clean.parquet"
silver_uri = None

if bucket is None:
    print("Skipping silver upload because the GCS bucket is not ready.")
elif "silver_path" not in globals() or not Path(silver_path).exists():
    print("Skipping silver upload because silver_path does not exist yet. Finish Part 5 first.")
else:
    try:
        silver_blob = bucket.blob(silver_blob_name)
        silver_blob.upload_from_filename(str(silver_path), content_type="application/octet-stream")
        silver_uri = f"gs://{GCS_BUCKET}/{silver_blob_name}"
        print("Uploaded silver parquet:", silver_uri)
        print("upload source:", silver_path)
    except Exception as exc:
        print("Silver upload did not work.")
        print("Check GCS permission and confirm the local silver parquet exists.")
        print("Original error:", exc)

print("silver cloud status:", silver_uri or "not uploaded")


### 7.6 Summarize Your Evidence

Use this cell for your PR note. It gives you a clean summary whether the cloud upload worked or not.


In [ ]:
print("Cloud evidence summary")
print("S3 source:", "ok" if s3_raw_bytes is not None else "blocked, local fallback used")
print("Bronze:", bronze_uri or "not uploaded, use blocker note")
print("Silver:", silver_uri or "not uploaded, use blocker note")

if bronze_uri and silver_uri:
    print("Cloud load evidence is ready for your PR.")
else:
    print("Cloud load is incomplete. Include the blocker message and your local evidence.")


### 7.7 Draw the Architecture You Just Built

Before you leave the notebook, draw a lightweight current-state architecture diagram. The goal is conceptual understanding, not design polish.

Your diagram may be a Mermaid diagram, draw.io sketch, slide, screenshot, or hand-drawn picture. It must show both paths:

1. Local rehearsal: local raw CSV to local bronze to local silver parquet.
2. Cloud run: public S3 raw object to GCS bronze to GCS silver parquet.

Label the source, bronze storage, transform step, silver storage, byte-for-byte proof, row-count proof, and any S3 or GCS blocker.


## Part 8 (Bonus After Instructor Approval): From Notebook to Pipeline Script

Do not start this until your notebook is complete and reviewed.

Once the notebook runs end to end, move your working code into your copy of `starter/medallion_etl.py` and run it like a real pipeline from your work folder:

```bash
cd student-work/week3/day3
uv run python medallion_etl.py --local
```

The script should print the same row counts the notebook produced (silver = 14). Keep functions small, add docstrings, and use a `__main__` guard.


## Evidence Checklist

Required evidence:

| Evidence | Where |
|---|---|
| Bronze file, byte-identical, in a dated folder | `data/bronze/weather/ingest_date=.../` |
| Silver parquet, 14 rows, typed | `data/silver/weather/weather_clean.parquet` |
| Bronze byte proof passed | Notebook output |
| Parquet round-trip proof passed | Notebook output |
| At least three cleaning policies explained | Your `student-work/week3/day3/README.md` or PR description |
| Short evidence summary | README or PR description |

Optional evidence:

| Evidence | Where |
|---|---|
| pandas and Polars row counts matching | Notebook outputs |
| S3 to GCS success, or a clear blocker note | PR description |
| Lightweight architecture diagram | README, PR description, screenshot, or photo |
| Script run output | Terminal paste in the PR |
